**Imports and setup**, including `utils` (spatial smoothing) and the hsiViewer.

In [ ]:
import numpy as np
import os
import spectral
from upwins_hsi import utils
from hsiViewer import hsi_viewer_layers as hlv

# --- Load configuration (paths + parameters live in config.yaml) ---
# config.yaml and the paths inside it are relative to the repo root, but this
# notebook lives in notebooks/. Walk up to the repo root and resolve every
# configured path against it, so this works whether Jupyter is launched from
# the repo root or from notebooks/.
import yaml
from pathlib import Path
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'config.yaml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
with open(REPO_ROOT / 'config.yaml') as _f:
    CONFIG = yaml.safe_load(_f)
for _section in ('paths',):
    for _key, _val in CONFIG.get(_section, {}).items():
        if isinstance(_val, str):
            CONFIG[_section][_key] = str(REPO_ROOT / _val)

**Load the calibration** (`gain`, `offset`) produced by notebook 01.

In [ ]:
# Read this collection's gain/offset, written by notebook 01 into calibration_dir.
# Nothing ships in the repo: run notebook 01 for THIS collection first so its
# calibration_dir holds gain.npy/offset.npy. Never point calibration_dir at
# another collection's bundle -- gain/offset absorb that collection's
# illumination and exposure and are not valid here.
import os
_cal_dir = CONFIG['paths']['calibration_dir']
gain = np.load(os.path.join(_cal_dir, 'gain.npy'))
offset = np.load(os.path.join(_cal_dir, 'offset.npy'))
print('Using calibration from: ' + _cal_dir)

## Open the Image to Convert to Reflectance

**Open a raw image to convert**, and read the smoothing level and bad-band ranges from the config. Set the image in `config.yaml`.

In [ ]:
# number of smoothing iterations
smoothing_level = CONFIG['reflectance']['smoothing_level']
# wl range(s) to remove (in nanometers)
bbl_wl_ranges = CONFIG['reflectance']['bbl_wl_ranges']

# raw image to convert to reflectance (set both entries in config.yaml)
raw_image_hdr = CONFIG['paths']['raw_image_hdr']
raw_image = CONFIG['paths']['raw_image']
# same_cube accepts BOTH header conventions -- <file>.hdr and <base>.hdr -- so a
# raw_5.bin cube pairs with either raw_5.bin.hdr or raw_5.hdr. Comparing bare
# Path().stem rejected the first, which is the convention the batch script uses,
# so the two entry points disagreed about the same collection.
from upwins_hsi import utils
assert utils.same_cube(raw_image, raw_image_hdr), \
    "raw_image and raw_image_hdr name different cubes — check config.yaml"
# Read the image
im = spectral.envi.open(raw_image_hdr, raw_image)
#im.Arr = im.load().astype(np.float32)
#im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

## Convert Image to Reflectance

**Convert to reflectance.** Drops bad bands, applies per-band gain/offset, masks empty pixels, spatially smooths, and saves the reflectance image next to the raw one.

In [ ]:
# ====== Convert to Reflectance ======
# This cell is idempotent: every value below is derived from pristine inputs --
# `im.wl` and the full-length gain/offset loaded above -- and nothing is rebound
# in place, so re-running it recomputes the same result. Re-run it as often as
# you like without re-running any earlier cell.
wl_all = np.asarray(im.wl)

# wl range(s) to remove (in nanometers)
bbl_wl_ranges = CONFIG['reflectance']['bbl_wl_ranges']
indices = []
for i in range(len(wl_all)):
    is_bad_band = False
    for bbl_wl_range in bbl_wl_ranges:
        if bbl_wl_range[0] < wl_all[i] < bbl_wl_range[1]:
            is_bad_band = True                
    if (not is_bad_band):
        indices.append(int(i))
indices = np.asarray(indices, dtype=np.int16)

# Band-grid guard: gain/offset are position-indexed and valid only for the
# band configuration they were fit on. Refuse up front if the calibration length
# does not match this image's band count, instead of misaligning silently or
# raising a confusing IndexError deep in the conversion loop below.
if len(gain) != len(wl_all):
    raise ValueError(
        f"Calibration has {len(gain)} bands but this image has {len(wl_all)}. "
        "gain.npy/offset.npy are only valid for the sensor/band configuration "
        "they were fit on -- re-run notebook 01 for this collection.")

# determine the parameters for the image
nr = im.nrows
nc = im.ncols
nb = len(indices)

# Good-band wavelengths, for the output header. gain and offset are deliberately
# NOT subset: the loop below indexes them by original band number, so they stay
# full-length and this cell never consumes its own output.
wl_good = wl_all[indices]

# Prepare an output array for the reflectance image
imRef = np.zeros((nr, nc, nb), dtype=np.float32)
# Create the data mask
mask = (im.read_band(0) > 0).astype(np.float32)
    
# ====== Load the image and compute reflectance ======
# Loop over the good bands and fill the result. `b` is the band's index into the
# raw cube AND into gain/offset -- one index for both, so a coefficient cannot
# drift out of step with the band it scales.
# reflectance = gain*counts + offset, matching notebook 01's empirical-line fit
# (LinearRegression with fit_intercept=True). The *mask is applied OUTSIDE the
# affine term so no-data pixels stay exactly 0 (offset must not leak into them),
# preserving the band0>0 mask convention used downstream.
print('Reading the image and converting to reflectance.')
for i, b in enumerate(indices):
    imRef[:, :, i] = ((gain[b]*np.squeeze(im.read_band(b)) + offset[b])*mask).astype(np.float32)
                            
# ====== Spatially smooth the image ======
for i in range(smoothing_level):
    print(f'Smoothing the image, iteration {i+1}.')
    imRef = utils.spatial_smoothing(imRef, mask=mask).astype(np.float32)

# ====== Save the image ======
# Save the image
print('Saving the image.')
# Copy the raw header rather than aliasing it: `md = im.metadata` edited the open
# image's own metadata in place, and this cell is meant to be re-runnable.
md = dict(im.metadata)
# Subset EVERY per-band key, not just wavelength. An ENVI header's wavelength,
# fwhm, band names and bbl are positional lists carrying one entry per band, so
# each must stay exactly `bands` long. Dropping bad bands used to shorten
# wavelength alone and leave the rest at the raw cube's full length, writing a
# _ref.hdr that declared one band count and then described another. Readers that
# trust those lists disagree with `bands` and mis-attribute wavelengths.
for _key in ('fwhm', 'band names', 'bbl'):
    _vals = md.get(_key)
    if _vals is None:
        continue
    if len(_vals) == len(wl_all):
        md[_key] = [_vals[b] for b in indices]
    else:
        print(f"  note: header key '{_key}' has {len(_vals)} entries for "
              f"{len(wl_all)} bands, so it was already inconsistent; left as is.")
md['wavelength'] = [str(w) for w in wl_good]
# `default bands` holds band NUMBERS for display. Dropping bands renumbers
# everything after the first gap, so any retained value would point at the wrong
# band; drop the hint rather than write a wrong one.
md.pop('default bands', None)
# Write next to the raw image as <raw_image>_ref.img/.hdr, derived straight from
# the raw_image config key (the single source of truth for this collection).
# save_image derives the .img name from the .hdr path; notebook 02's viewer cell
# below and notebook 03's default both open this same <raw_image>_ref product.
# reflectance_paths drops a known ENVI data extension before appending _ref, so
# a raw_5.bin cube writes raw_5_ref.img rather than raw_5.bin_ref.img. It is the
# single definition of this name -- the viewer cell below and notebook 03's
# default read it from the same helper, so they cannot look for a name this cell
# did not write.
_ref_hdr, _ref_img = utils.reflectance_paths(CONFIG['paths']['raw_image'])
spectral.envi.save_image(_ref_hdr, imRef, metadata=md, force=True)

**Optional — open a reflectance image** to inspect it.

In [ ]:
# Reflectance image to view: notebook 02 always views its OWN output -- the _ref
# product the save cell just wrote next to raw_image. Derive that path from
# raw_image (the single source of truth) instead of re-reading a separate config
# key, so this cell can never open a different cube than was just written. (The
# reflectance_image keys in config.yaml are notebook 03's input, not used here.)
reflectance_image_hdr, reflectance_image = utils.reflectance_paths(
    CONFIG['paths']['raw_image'])
# Read the image
im = spectral.envi.open(reflectance_image_hdr, reflectance_image)
im.Arr = im.load().astype(np.float32)
im.mask = im.Arr[:,:,0]!=0
im.wl = np.asarray(im.bands.centers)
wl = im.wl

**Interactive.** Opens the reflectance image in the hsiViewer to examine pixels and spectra.

In [ ]:
# If you want to manually examine the image and spectra
hlv.viewer(im)